# 05 -- Representational Geometry and CKA

Analyze the geometry of residual stream activations: do conflict and control items
occupy distinct regions of representation space? How does this geometry change
across layers and between standard vs reasoning models?

**What this notebook does:**
1. Cosine silhouette score (primary separation metric) with bootstrap CIs
2. Permutation test for significance of cluster separation
3. PCA / UMAP projections with random projection control
4. Layer-wise CKA between matched-architecture model pairs
5. Intrinsic dimensionality (Two-NN + participation ratio)
6. Linear separability with PCA pre-reduction (d >> N control)

**Modes:**
- **Synthetic** (default): planted signal at layer 2 (dim 0)
- **Real**: switch `DATA_PATH`

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from s1s2.utils.io import (
    open_activations, get_residual, load_problem_metadata,
    list_models, model_metadata,
)
from s1s2.geometry.clusters import (
    cosine_silhouette, cosine_silhouette_with_ci,
    silhouette_permutation_test, calinski_harabasz, davies_bouldin,
)
from s1s2.geometry.projections import pca_project, random_projection
from s1s2.geometry.intrinsic_dim import two_nn_intrinsic_dim, participation_ratio
from s1s2.geometry.cka import linear_cka
from s1s2.viz.theme import set_paper_theme, COLORS

set_paper_theme()

In [ ]:
# ---- Mode selection ----
USE_SYNTHETIC = True  # Set to False for real data

if USE_SYNTHETIC:
    from tests.conftest import build_synthetic_hdf5, SYNTH_MODEL_KEY
    DATA_PATH = REPO_ROOT / "data" / "activations" / "synthetic_nb05.h5"
    build_synthetic_hdf5(DATA_PATH)
    MODEL_KEY = SYNTH_MODEL_KEY
    POSITION = "P0"
    print(f"Built synthetic HDF5 at {DATA_PATH}")
    print("Planted signal: +0.8 shift on dim 0 at layer 2 for conflict items.")
else:
    DATA_PATH = REPO_ROOT / "data" / "activations" / "main.h5"  # REQUIRES REAL DATA
    MODEL_KEY = "meta-llama_Llama-3.1-8B-Instruct"  # REQUIRES REAL DATA
    POSITION = "P0"
    print(f"Using real data at {DATA_PATH}")

In [ ]:
# --- Load metadata ---
with open_activations(DATA_PATH) as f:
    meta = load_problem_metadata(f)
    mm = model_metadata(f, MODEL_KEY)
    n_layers = int(mm["n_layers"])

conflict = meta["conflict"]
labels = conflict.astype(np.int32)  # 1=conflict, 0=control

print(f"Model: {MODEL_KEY}")
print(f"Layers: {n_layers}, Hidden: {mm['hidden_dim']}")
print(f"Problems: {len(conflict)} (conflict={conflict.sum()}, control={(~conflict).sum()})")

silhouette_results = []

with open_activations(DATA_PATH) as f:
    for layer in range(n_layers):
        X = get_residual(f, MODEL_KEY, layer, position=POSITION).astype(np.float64)

        # Cosine silhouette with bootstrap CI
        sil, ci_lo, ci_hi = cosine_silhouette_with_ci(
            X, labels, n_bootstrap=200, seed=42
        )

        # Permutation test: returns (observed, p_value, null_distribution)
        _, perm_p, _ = silhouette_permutation_test(
            X, labels, n_permutations=200, seed=42
        )

        # Additional metrics
        ch = calinski_harabasz(X, labels)
        db = davies_bouldin(X, labels)

        silhouette_results.append({
            "layer": layer,
            "silhouette": sil,
            "ci_lo": ci_lo,
            "ci_hi": ci_hi,
            "perm_p": perm_p,
            "calinski_harabasz": ch,
            "davies_bouldin": db,
        })
        print(f"  Layer {layer:2d}: sil={sil:.4f} [{ci_lo:.4f}, {ci_hi:.4f}], "
              f"perm_p={perm_p:.4f}")

df_sil = pd.DataFrame(silhouette_results)

In [ ]:
silhouette_results = []

with open_activations(DATA_PATH) as f:
    for layer in range(n_layers):
        X = get_residual(f, MODEL_KEY, layer, position=POSITION).astype(np.float64)

        # Cosine silhouette with bootstrap CI
        sil, ci_lo, ci_hi = cosine_silhouette_with_ci(
            X, labels, n_bootstrap=200, seed=42
        )

        # Permutation test
        perm_result = silhouette_permutation_test(
            X, labels, n_permutations=200, seed=42
        )

        # Additional metrics
        ch = calinski_harabasz(X, labels)
        db = davies_bouldin(X, labels)

        silhouette_results.append({
            "layer": layer,
            "silhouette": sil,
            "ci_lo": ci_lo,
            "ci_hi": ci_hi,
            "perm_p": perm_result.p_value,
            "calinski_harabasz": ch,
            "davies_bouldin": db,
        })
        print(f"  Layer {layer:2d}: sil={sil:.4f} [{ci_lo:.4f}, {ci_hi:.4f}], "
              f"perm_p={perm_result.p_value:.4f}")

df_sil = pd.DataFrame(silhouette_results)

In [ ]:
# PCA projection
proj_pca, ev_ratio = pca_project(X_peak, n_components=2, seed=42)

# Random projections (100 random 2D projections as control)
# random_projection returns a list of (n, 2) arrays
rp_list = random_projection(X_peak, n_components=2, n_seeds=100, seed=42)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# PCA
for lbl, color, name in [(1, COLORS["s1"], "Conflict"), (0, COLORS["s2"], "Control")]:
    mask = labels == lbl
    axes[0].scatter(proj_pca[mask, 0], proj_pca[mask, 1],
                    c=color, alpha=0.6, s=20, label=name)
axes[0].set_xlabel(f"PC1 ({ev_ratio[0]:.1%})")
axes[0].set_ylabel(f"PC2 ({ev_ratio[1]:.1%})")
axes[0].set_title(f"PCA (layer {peak_layer})")
axes[0].legend(fontsize=8)

# UMAP (optional -- may not be installed)
try:
    from s1s2.geometry.projections import umap_project
    proj_umap = umap_project(X_peak, n_components=2, seed=42)
    for lbl, color, name in [(1, COLORS["s1"], "Conflict"), (0, COLORS["s2"], "Control")]:
        mask = labels == lbl
        axes[1].scatter(proj_umap[mask, 0], proj_umap[mask, 1],
                        c=color, alpha=0.6, s=20, label=name)
    axes[1].set_xlabel("UMAP 1")
    axes[1].set_ylabel("UMAP 2")
    axes[1].set_title(f"UMAP (layer {peak_layer})")
    axes[1].legend(fontsize=8)
except ImportError:
    axes[1].text(0.5, 0.5, "umap-learn not installed\n(pip install umap-learn)",
                 ha="center", va="center", transform=axes[1].transAxes)
    axes[1].set_title("UMAP (not available)")

# Random projection: show one example
rp_one = rp_list[0]  # (n, 2)
for lbl, color, name in [(1, COLORS["s1"], "Conflict"), (0, COLORS["s2"], "Control")]:
    mask = labels == lbl
    axes[2].scatter(rp_one[mask, 0], rp_one[mask, 1],
                    c=color, alpha=0.6, s=20, label=name)
axes[2].set_xlabel("Random dim 1")
axes[2].set_ylabel("Random dim 2")
axes[2].set_title(f"Random projection (1 of {len(rp_list)})")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Silhouette on random projections vs PCA
sil_pca = cosine_silhouette(proj_pca, labels)
sil_rps = [cosine_silhouette(rp, labels) for rp in rp_list[:100]]
print(f"\nSilhouette (PCA 2D): {sil_pca:.4f}")
print(f"Silhouette (random 2D): mean={np.mean(sil_rps):.4f}, "
      f"std={np.std(sil_rps):.4f}")
if sil_pca > np.mean(sil_rps) + 2 * np.std(sil_rps):
    print("PCA structure is likely genuine (exceeds random projection envelope).")
else:
    print("Caution: PCA structure may be an artifact (within random projection envelope).")

## PCA Projection + Random Projection Control

Visualize the representation at the peak silhouette layer. The random projection
control checks whether PCA/UMAP-visible structure is genuine or an artifact of
the projection method.

In [ ]:
# Pick the peak layer
peak_layer = int(df_sil.loc[df_sil["silhouette"].idxmax(), "layer"])
print(f"Peak silhouette layer: {peak_layer} (sil={df_sil.loc[df_sil['silhouette'].idxmax(), 'silhouette']:.4f})")

with open_activations(DATA_PATH) as f:
    X_peak = get_residual(f, MODEL_KEY, peak_layer, position=POSITION).astype(np.float64)

In [ ]:
# PCA projection
proj_pca, ev_ratio = pca_project(X_peak, n_components=2, seed=42)

# Random projections (100 random 2D projections as control)
rp_panel = random_projection(X_peak, n_components=2, n_projections=100, seed=42)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# PCA
for lbl, color, name in [(1, COLORS["s1"], "Conflict"), (0, COLORS["s2"], "Control")]:
    mask = labels == lbl
    axes[0].scatter(proj_pca[mask, 0], proj_pca[mask, 1],
                    c=color, alpha=0.6, s=20, label=name)
axes[0].set_xlabel(f"PC1 ({ev_ratio[0]:.1%})")
axes[0].set_ylabel(f"PC2 ({ev_ratio[1]:.1%})")
axes[0].set_title(f"PCA (layer {peak_layer})")
axes[0].legend(fontsize=8)

# UMAP (optional -- may not be installed)
try:
    from s1s2.geometry.projections import umap_project
    proj_umap = umap_project(X_peak, n_components=2, seed=42)
    for lbl, color, name in [(1, COLORS["s1"], "Conflict"), (0, COLORS["s2"], "Control")]:
        mask = labels == lbl
        axes[1].scatter(proj_umap[mask, 0], proj_umap[mask, 1],
                        c=color, alpha=0.6, s=20, label=name)
    axes[1].set_xlabel("UMAP 1")
    axes[1].set_ylabel("UMAP 2")
    axes[1].set_title(f"UMAP (layer {peak_layer})")
    axes[1].legend(fontsize=8)
except ImportError:
    axes[1].text(0.5, 0.5, "umap-learn not installed\n(pip install umap-learn)",
                 ha="center", va="center", transform=axes[1].transAxes)
    axes[1].set_title("UMAP (not available)")

# Random projection: show one example
rp_one = rp_panel.projections[0]  # (n, 2)
for lbl, color, name in [(1, COLORS["s1"], "Conflict"), (0, COLORS["s2"], "Control")]:
    mask = labels == lbl
    axes[2].scatter(rp_one[mask, 0], rp_one[mask, 1],
                    c=color, alpha=0.6, s=20, label=name)
axes[2].set_xlabel("Random dim 1")
axes[2].set_ylabel("Random dim 2")
axes[2].set_title(f"Random projection (1 of {rp_panel.n_projections})")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Silhouette on random projections vs PCA
sil_pca = cosine_silhouette(proj_pca, labels)
sil_rps = [cosine_silhouette(rp_panel.projections[i], labels)
           for i in range(min(100, rp_panel.n_projections))]
print(f"\nSilhouette (PCA 2D): {sil_pca:.4f}")
print(f"Silhouette (random 2D): mean={np.mean(sil_rps):.4f}, "
      f"std={np.std(sil_rps):.4f}")
if sil_pca > np.mean(sil_rps) + 2 * np.std(sil_rps):
    print("PCA structure is likely genuine (exceeds random projection envelope).")
else:
    print("Caution: PCA structure may be an artifact (within random projection envelope).")

## Intrinsic Dimensionality

Two complementary estimators:
- **Two-NN** (Facco et al. 2017): local manifold dimension
- **Participation ratio** (Gao et al. 2017): effective eigenspectrum dimension

In [ ]:
id_results = []

with open_activations(DATA_PATH) as f:
    for layer in range(n_layers):
        X = get_residual(f, MODEL_KEY, layer, position=POSITION).astype(np.float64)

        # Two-NN for all items, conflict only, control only
        id_all = two_nn_intrinsic_dim(X)
        id_conflict = two_nn_intrinsic_dim(X[conflict]) if conflict.sum() >= 5 else np.nan
        id_control = two_nn_intrinsic_dim(X[~conflict]) if (~conflict).sum() >= 5 else np.nan

        # Participation ratio
        pr_all = participation_ratio(X)

        id_results.append({
            "layer": layer,
            "two_nn_all": id_all,
            "two_nn_conflict": id_conflict,
            "two_nn_control": id_control,
            "participation_ratio": pr_all,
        })

df_id = pd.DataFrame(id_results)
print(df_id.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Two-NN
axes[0].plot(df_id["layer"], df_id["two_nn_all"], "o-",
             color=COLORS["standard"], label="All", markersize=5)
axes[0].plot(df_id["layer"], df_id["two_nn_conflict"], "s--",
             color=COLORS["s1"], label="Conflict", markersize=5)
axes[0].plot(df_id["layer"], df_id["two_nn_control"], "^--",
             color=COLORS["s2"], label="Control", markersize=5)
axes[0].set_xlabel("Layer")
axes[0].set_ylabel("Two-NN intrinsic dim")
axes[0].set_title("Two-NN intrinsic dimensionality")
axes[0].legend(fontsize=8)

# Participation ratio
axes[1].plot(df_id["layer"], df_id["participation_ratio"], "o-",
             color=COLORS["reasoning"], markersize=5)
axes[1].set_xlabel("Layer")
axes[1].set_ylabel("Participation ratio")
axes[1].set_title("Eigenspectrum effective dimension")

plt.tight_layout()
plt.show()

## Layer-Matched CKA

Compare representations between two models at corresponding layers.
Primary use: how similar are Llama-3.1-8B-Instruct and R1-Distill-Llama-8B?

In [ ]:
with open_activations(DATA_PATH) as f:
    all_models = list_models(f)

if len(all_models) >= 2:
    model_a, model_b = all_models[0], all_models[1]
    print(f"Computing CKA: {model_a} vs {model_b}")

    cka_results = []
    n_layers_a = int(model_metadata(f, model_a)["n_layers"])
    n_layers_b = int(model_metadata(f, model_b)["n_layers"])
    n_common = min(n_layers_a, n_layers_b)

    with open_activations(DATA_PATH) as f:
        for layer in range(n_common):
            X_a = get_residual(f, model_a, layer, position=POSITION).astype(np.float64)
            X_b = get_residual(f, model_b, layer, position=POSITION).astype(np.float64)
            cka_val = linear_cka(X_a, X_b)
            cka_results.append({"layer": layer, "cka": cka_val})

    df_cka = pd.DataFrame(cka_results)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df_cka["layer"], df_cka["cka"], "o-", color=COLORS["standard"], markersize=5)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Linear CKA")
    ax.set_title(f"CKA: {model_a} vs {model_b}")
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()
else:
    print(f"Only one model in dataset ({all_models}). CKA requires two models.")
    print("\nWith real data, this cell computes within-model CKA instead:")
    print("  CKA(layer_i, layer_j) for all i,j -- reveals which layers share structure.")

    # Within-model CKA as fallback
    print(f"\nComputing within-model CKA matrix for {MODEL_KEY}...")
    cka_matrix = np.zeros((n_layers, n_layers))
    activations_cache = {}
    with open_activations(DATA_PATH) as f:
        for layer in range(n_layers):
            activations_cache[layer] = get_residual(
                f, MODEL_KEY, layer, position=POSITION
            ).astype(np.float64)

    for i in range(n_layers):
        for j in range(i, n_layers):
            cka_val = linear_cka(activations_cache[i], activations_cache[j])
            cka_matrix[i, j] = cka_val
            cka_matrix[j, i] = cka_val

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cka_matrix, cmap="viridis", vmin=0, vmax=1)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Layer")
    ax.set_title(f"Within-model CKA: {MODEL_KEY}")
    plt.colorbar(im, ax=ax, label="Linear CKA")
    plt.tight_layout()
    plt.show()

## PCA Explained Variance Spectrum

In [ ]:
# PCA spectrum at peak layer
n_components_to_check = min(20, X_peak.shape[0] - 1, X_peak.shape[1])
_, ev_full = pca_project(X_peak, n_components=n_components_to_check, seed=42)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(range(len(ev_full)), ev_full, color=COLORS["standard"], edgecolor="white")
axes[0].set_xlabel("Principal component")
axes[0].set_ylabel("Explained variance ratio")
axes[0].set_title(f"PCA spectrum (layer {peak_layer})")

axes[1].plot(range(len(ev_full)), np.cumsum(ev_full), "o-",
             color=COLORS["standard"], markersize=4)
axes[1].axhline(0.95, color="red", linestyle="--", alpha=0.5, label="95%")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative explained variance")
axes[1].set_title("Cumulative PCA spectrum")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

n_95 = int(np.searchsorted(np.cumsum(ev_full), 0.95)) + 1
print(f"Components for 95% variance: {n_95}/{n_components_to_check}")

## Summary

**Key results:**

1. **Cosine silhouette**: which layer(s) show significant separation between conflict and control representations? Compare with probing peak (notebook 02).
2. **Random projection control**: if UMAP shows clusters but random projections do not, UMAP is hallucinating structure.
3. **Intrinsic dimensionality**: do conflict and control items live on different-dimensional manifolds? A difference suggests the model allocates different representational resources.
4. **CKA**: how does reasoning distillation change the layer-wise representation? A CKA dip at a particular layer suggests that layer diverged most during distillation.

**Caveats (per CLAUDE.md):**
- **d >> N pitfall**: with 4096 dims and ~500 points, random classes are linearly separable (Cover's theorem). Always PCA-reduce to 50-100 dims before SVM/logistic separability.
- **Random projection baseline**: mandatory before trusting UMAP.
- **Cosine over Euclidean**: Euclidean distances concentrate in high-d, making Euclidean silhouette uninformative.